In [5]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Load dataset
iris = load_iris()

X = iris.data
y = iris.target

print("Dataset shape:", X.shape)
print("Classes:", iris.target_names)


# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


# Calculate mean and standard deviation
def calculate_mean_std(X, y):
    mean = {}
    std = {}

    classes = np.unique(y)

    for c in classes:

        X_class = X[y == c]

        mean[c] = np.mean(X_class, axis=0)

        # Add smoothing value
        std[c] = np.std(X_class, axis=0) + 1e-9

    return mean, std


mean, std = calculate_mean_std(X_train, y_train)


# Calculate prior probability
def calculate_prior(y):

    classes, counts = np.unique(y, return_counts=True)

    prior = {}

    total = len(y)

    for c, count in zip(classes, counts):

        # Smoothing for prior probability
        prior[c] = (count + 1) / (total + len(classes))

    return prior


prior = calculate_prior(y_train)


# Gaussian probability with smoothing
def gaussian_probability(x, mean, std):

    epsilon = 1e-9

    std = std + epsilon

    probability = (
        1 / (np.sqrt(2 * np.pi) * std)
    ) * np.exp(
        -((x - mean) ** 2) / (2 * std ** 2)
    )

    # Prevent probability from becoming zero
    probability = max(probability, epsilon)

    return probability


# Prediction
def predict_sample(sample):

    classes = np.unique(y_train)

    probabilities = {}

    for c in classes:

        # Start with prior probability
        prob = prior[c]

        # Calculate likelihood
        for i in range(len(sample)):

            prob *= gaussian_probability(
                sample[i],
                mean[c][i],
                std[c][i]
            )

        probabilities[c] = prob

    return max(probabilities, key=probabilities.get)


# Predict test data
y_pred = []

for sample in X_test:

    prediction = predict_sample(sample)

    y_pred.append(prediction)


y_pred = np.array(y_pred)


# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("\nActual values:")
print(y_test)

print("\nPredicted values:")
print(y_pred)

print("\nAccuracy:", accuracy)


# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

Dataset shape: (150, 4)
Classes: ['setosa' 'versicolor' 'virginica']
Training data: (120, 4)
Testing data: (30, 4)

Actual values:
[1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]

Predicted values:
[1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]

Accuracy: 1.0

Confusion Matrix:
[[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
